In [1]:
import pandas as pd
import os
from transformers import AutoTokenizer
import torch


def print_from_csv(csv_path, tokenizer):
    """
    Read the PCA+L2+HDBSCAN result CSV and print the top-20 tokens of each cluster,
    and compute the mean probability of each cluster, as well as the global mean/std
    (excluding the -1 cluster).
    The CSV must contain: cluster_id, token_id, probability
    """
    df = pd.read_csv(csv_path)
    if not {"cluster_id", "token_id", "probability"} <= set(df.columns):
        raise ValueError("CSV must contain cluster_id, token_id, probability columns")

    # ===== noise statistics =====
    total_tokens = len(df)
    noise_df = df[df["cluster_id"] == -1]
    noise_tokens = len(noise_df)
    noise_ratio = noise_tokens / total_tokens * 100 if total_tokens > 0 else 0.0


    # Remove -1 cluster before computing global mean/std
    df_no_noise = df[df["cluster_id"] != -1]
    global_mean = df_no_noise["probability"].mean()
    global_std = df_no_noise["probability"].std(ddof=0)  # Population standard deviation

    # Group by cluster (excluding -1)
    clusters = sorted(set(df["cluster_id"]) - {-1})
    for cid in clusters:
        sub = df[df["cluster_id"] == cid]
        mean_prob = sub["probability"].mean()
        # Take the top 20 tokens in descending order of probability
        top = sub.sort_values("probability", ascending=False).head(100)
        tokens = []
        for tid in top["token_id"].astype(int).tolist():
            #print(tid)
            try:
                tok_str = tokenizer.decode([tid])
            except Exception:
                tok_str = tokenizer.convert_ids_to_tokens([tid])[0]
            tokens.append(tok_str)
        print(f"Cluster {cid} (size={len(sub)}, mean_prob={mean_prob:.4f}):")
        print(", ".join(tokens))
        print()

    print(
        f"Global probability mean = {global_mean:.4f}, "
        f"std = {global_std:.4f} (excluding noise)"
        f"\nNoise tokens: {noise_tokens} / {total_tokens} "
        f"({noise_ratio:.2f}%)"
    )


In [2]:

model_name = "mistralai/Mistral-7B-v0.1"
# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

print(f"[✓] Tokenizer loaded from: {tokenizer_path}")

# ===== Step 2: Load lm_head.weight =====
space_name = "output_proj"
lm_head_path = f"{model_name}/tensors/{space_name}.pt" 
embedding_matrix = torch.load(lm_head_path, map_location="cpu")
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")
embedding_matrix = embedding_matrix.cpu().to(torch.float32).numpy()
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")

[✓] Tokenizer loaded from: mistralai/Mistral-7B-v0.1/tokenizer
[✓] lm_head.weight loaded: shape = torch.Size([32000, 4096])
[✓] lm_head.weight loaded: shape = (32000, 4096)


In [3]:
data_path = "comp/mistralai/Mistral-7B-v0.1/seed_42"
csv_path = os.path.join(data_path, "output_proj_l2=False.csv")

In [7]:
print_from_csv(csv_path, tokenizer)

Cluster 0 (size=10, mean_prob=0.9502):
item, Item, item, items, items, Item, Items, ITEM, getItem, ITE

Cluster 1 (size=9, mean_prob=0.9368):
replace, replaced, replace, Replace, replacing, replacement, 替, substitute, subst

Cluster 2 (size=7, mean_prob=0.9731):
cancel, cancell, cancel, cancelled, Cancel, cancellation, ancell

Cluster 3 (size=13, mean_prob=0.9437):
update, update, Update, updating, updates, 更, Update, updated, UPDATE, updated, Updated, Updated, upgrade

Cluster 4 (size=9, mean_prob=0.9522):
turn, turn, turned, Turn, Turn, turns, turning, turno, Turner

Cluster 5 (size=10, mean_prob=0.9603):
path, path, Path, Path, paths, PATH, 路, paths, Paths, pathy

Cluster 6 (size=10, mean_prob=0.9664):
program, Program, Program, program, програ, programa, programs, programme, 程, programming

Cluster 7 (size=8, mean_prob=0.9727):
Entity, entity, entity, entities, entities, Entity, Entities, ENT

Cluster 8 (size=12, mean_prob=0.9577):
License, license, license, License, licenses, Lic,

In [14]:
out_root = "comp"
model_name = "gpt-oss"
space_name = "output_proj"

# ===== Step 1: Load tokenizer =====
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"[✓] Tokenizer loaded from: {tokenizer_path}")


[✓] Tokenizer loaded from: gpt-oss/tokenizer


In [16]:
print_from_csv("comp/gpt-oss/output_proj/clusters_l2=True/run_00001_clusters.csv", tokenizer)

Cluster 0 (size=21, mean_prob=0.6669):
	D, 	G, 	R, 	F, 	M, 	L, 	H, 	P, 	B, 	T, 	V, 	E, 	N, 	I, 	A, 	C, 	J, 	K, 	U, 	W

Cluster 1 (size=5, mean_prob=1.0000):
αλ, πτ, //*[@, αρ, βε

Cluster 2 (size=10, mean_prob=0.9933):
’A, ’E, ’O, ’H, .bluetooth, ’U, .userid, .datab, .cloudflare, ®,

Cluster 3 (size=5, mean_prob=1.0000):
/android, &i, .yahoo, /mysql, /github

Cluster 4 (size=31, mean_prob=0.8619):
	mp, 	sf, 	dp, 	fd, 	mt, 	td, 	md, 	pp, 	fp, 	rt, 	cc, 	pm, 	ds, 	pc, 	lp, 	ct, 	tc, 	ws, 	cv, 	vm

Cluster 5 (size=27, mean_prob=0.7443):
П, К, Н, Т, З, У, А, М, Д, О, Г, И, Я, Ф, Р, Х, Б, Е, Л, Ш

Cluster 6 (size=7, mean_prob=0.9760):
<IAction, -क, .osgi, ,module, :[", -म, -अ

Cluster 7 (size=7, mean_prob=0.9857):
 beiden,  eigenen,  richtigen,  entsprechenden,  einfachen,  höheren,  gewünsch

Cluster 8 (size=10, mean_prob=0.9462):
-П, -К, -М, -Д, -Т, -А, -Б, -Ф, -С, .aliy

Cluster 9 (size=6, mean_prob=1.0000):
_producto, _empresa, _firestore, .flutter, 'Union, _produk

Cluster 10 (size=5, 

In [6]:
print_from_csv("comp/gpt-oss/output_proj/clusters_l2=True/run_00002_clusters.csv", tokenizer)

Cluster 0 (size=10, mean_prob=0.9839):
 vpraš,  spørg,  spørgsmål,  spørsmål,  pitanje,  pitanja,  frågor,  spør,  küsim,  frå

Cluster 1 (size=8, mean_prob=0.9805):
.ค, .ย, .ส, .พ, .ต, .อ, .ศ, ศก

Cluster 2 (size=48, mean_prob=0.9008):
�, �, �, �, �, �, �, �, �, �, �, �, �, �, �, �, �, �, �, �

Cluster 3 (size=5, mean_prob=1.0000):
 cush,  cushions,  Curtain,  Cushion,  Corridor

Cluster 4 (size=20, mean_prob=0.9220):
.В, .С, .А, .П, .К, .И, .Н, .О, .Д, .Т, .М, .Б, .Г, .к, .п, .д, .м, .б, .е, "А

Cluster 5 (size=7, mean_prob=0.9999):
 escort,  escorts,  Escort, Escort, escort,  eskort,  escorted

Cluster 6 (size=362, mean_prob=0.9904):
 ",
, );

/, 
, ;
, );
, 

,  {
, )
,  ];
,  #
,  {}

, );
/, {
//, __
, )");
, ...");
,  ''
,  "")
, :
//, $


Cluster 7 (size=5, mean_prob=1.0000):
 coat,  coats,  carpets,  cages,  collars

Cluster 8 (size=5, mean_prob=1.0000):
 tattoo,  tattoos,  Tattoo,  Tarot,  Tattoos

Cluster 9 (size=5, mean_prob=1.0000):
 bless,  blessing,  Bless,  blessings,  

In [8]:
print_from_csv("comp/gpt-oss/output_proj/clusters_l2=True/run_00009_clusters.csv", tokenizer)

26638
21081
31038
69269
104352
87634
76700
9509
195640
84260
8041
180508
140976
6002
Cluster 0 (size=14, mean_prob=0.9009):
Generic,  generic,  Generic, generic, _generic, .generic, _GENERIC, .Generic, (Generic, _Generic, eneric,  泛, 泛, �

16109
46198
48934
80341
187649
145798
171957
143350
114007
182509
Cluster 1 (size=10, mean_prob=0.9521):
 tank,  Tank,  tanks, Tank, tank,  Tanks,  tanque,  tanker, 坦,  tanke

9641
13267
14362
17376
17665
24254
25269
26795
67744
33898
40343
62738
69186
85712
76636
74355
152643
169932
127846
87413
Cluster 2 (size=32, mean_prob=0.9679):
 flag, Flag, Flags,  flags, flag, flags, _flag, _FLAG, 	flag, _flags,  Flag, .flags, (flag, FLAG,  Flags, _FLAGS, 	flags, .Flag, .Flags, .flag

25324
85417
198675
159668
171316
170392
86909
182217
3066
87815
Cluster 3 (size=10, mean_prob=0.9697):
 blind,  Blind,  blinded,  blindness, blind, Blind,  blinds,  blindly, �,  deaf

79183
121033
126511
161232
159051
161864
66434
Cluster 4 (size=7, mean_prob=0.9754):
 Scout,  s

In [12]:
print_from_csv("comp/mistralai/Mixtral-8x7B-v0.1/output_proj/clusters_l2=True/run_00001_clusters.csv", tokenizer)

Cluster 0 (size=10, mean_prob=0.9022):
июля, января, июня, апреля, октября, февраля, сентября, декабря, августа, ноября

Cluster 1 (size=5, mean_prob=1.0000):
apt, oz, gn, yt, hal

Cluster 2 (size=5, mean_prob=1.0000):
ande, üt, utz, icher, ahren

Cluster 3 (size=7, mean_prob=0.9942):
somet, parsed, cached, decode, encode, opts, serialized

Cluster 4 (size=10, mean_prob=0.9911):
高, 手, 整, 集, 先, 引, 基, 格, 外, 好

Cluster 5 (size=5, mean_prob=1.0000):
мет, mixer, soap, bias, staff

Cluster 6 (size=11, mean_prob=0.9719):
ents, ries, ains, ances, ila, ored, ues, iers, aly, ices, aling

Cluster 7 (size=10, mean_prob=0.9993):
flags, handle, queue, property, plugin, setup, submit, controller, database, thread

Cluster 8 (size=9, mean_prob=0.9707):
janvier, avril, juin, juillet, septembre, décembre, octobre, février, novembre

Cluster 9 (size=15, mean_prob=0.9148):
ด, ต, ข, แ, ไ, ค, ก, ผ, ใ, ช, ห, น, จ, ว, 首

Cluster 10 (size=11, mean_prob=0.9579):
ל, न, र, त, म, מ, ת, ש, स, द, ह

Cluster 11 (size

In [13]:
print_from_csv("comp/mistralai/Mixtral-8x7B-v0.1/output_proj/clusters_l2=True/run_00002_clusters.csv", tokenizer)

Cluster 0 (size=9, mean_prob=0.9726):
Џ, Ћ, Ґ, Љ, Ї, Њ, Ё, Ђ, Щ

Cluster 1 (size=10, mean_prob=0.9505):
ию, окт, апре, сент, янва, авгу, дека, міс, февра, області

Cluster 2 (size=6, mean_prob=0.9996):
Cache, cache, CACHE, cached, Cache, cache

Cluster 3 (size=5, mean_prob=1.0000):
parse, Parse, Parse, Pars, Pars

Cluster 4 (size=15, mean_prob=0.9600):
Buffer, buff, Buff, Buff, Buffalo, buff, buffer, Buffer, buffers, buffer, buf, buf, BUFFER, Buf, BUF

Cluster 5 (size=8, mean_prob=0.9810):
algorithm, alg, algorithms, algorithm, 算, Algorithm, alg, algo

Cluster 6 (size=7, mean_prob=0.9814):
cancel, cancell, cancel, cancelled, Cancel, cancellation, ancel

Cluster 7 (size=10, mean_prob=0.9654):
ali, Ali, ali, alias, alias, Ali, Alias, али, alien, alis

Cluster 8 (size=6, mean_prob=1.0000):
anim, anim, Anim, animation, animated, Anim

Cluster 9 (size=6, mean_prob=1.0000):
pipe, pipe, pipeline, Pipeline, pipeline, pip

Cluster 10 (size=7, mean_prob=0.9999):
pad, pad, padding, пад, Pad, Pad,

In [4]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00009_clusters.csv", tokenizer)

Cluster 0 (size=8, mean_prob=0.9573):
device, device, Device, devices, Device, devices, DEVICE, 裝

Cluster 1 (size=10, mean_prob=0.9447):
Engine, Engine, engineers, engine, engineer, engineering, Engineering, engine, engines, 엔

Cluster 2 (size=13, mean_prob=0.9673):
Service, service, service, Services, Service, services, Services, services, ervice, SERVICE, служ, ervices, слу

Cluster 3 (size=7, mean_prob=0.9995):
ὸ, ῆ, ῶ, ὰ, ὴ, ὶ, ή

Cluster 4 (size=16, mean_prob=0.9780):
ления, жения, чения, шения, нения, жение, чение, нение, ление, шение, вания, ждения, ження, лення, чення, zenia

Cluster 5 (size=8, mean_prob=0.9651):
Dele, deleg, Delegate, deleg, delegate, delegate, 委, deck

Cluster 6 (size=16, mean_prob=0.9103):
check, Check, Check, checked, checking, check, checks, CHECK, CHECK, checked, 检, Checked, Checker, checkbox, chk, 체

Cluster 7 (size=9, mean_prob=0.9444):
channel, Channel, channel, Channel, channels, channels, CHANNEL, annels, chan

Cluster 8 (size=8, mean_prob=0.9802):


In [4]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00002_clusters.csv", tokenizer)

Cluster 0 (size=5, mean_prob=1.0000):
ipher, iper, iop, iom, ioc

Cluster 1 (size=7, mean_prob=0.9990):
декабря, октября, апреля, февраля, июня, ноября, июля

Cluster 2 (size=5, mean_prob=1.0000):
riz, rez, pez, lez, bez

Cluster 3 (size=5, mean_prob=1.0000):
fuck, shit, fucking, damn, bullshit

Cluster 4 (size=6, mean_prob=1.0000):
xF, xC, fx, xE, xA, xD

Cluster 5 (size=8, mean_prob=0.9964):
eme, edi, emat, emen, emi, eli, ем, emy

Cluster 6 (size=6, mean_prob=0.9996):
ERR, ERN, erra, eria, erna, elia

Cluster 7 (size=7, mean_prob=0.9983):
ological, omic, otic, etic, ographic, ometric, ometry

Cluster 8 (size=8, mean_prob=0.9976):
RES, REE, REF, RET, REC, reset, RESET, RE

Cluster 9 (size=5, mean_prob=1.0000):
chunk, Charlie, chunk, Chuck, chunks

Cluster 10 (size=8, mean_prob=0.9995):
properties, parameters, functions, methods, categories, conditions, Functions, Modules

Cluster 11 (size=11, mean_prob=0.9860):
ility, ivity, itivity, ativity, icity, idity, ibility, inity, urity, uity

In [15]:
print_from_csv("comp/gpt-oss/output_proj/random/seed_42/clusters_l2=True/run_00001_clusters.csv", tokenizer)

Cluster 0 (size=22, mean_prob=0.6451):
	G, 	D, 	L, 	R, 	F, 	M, 	H, 	P, 	B, 	T, 	V, 	E, 	N, 	I, 	A, 	C, 	K, 	J, 	W, 	O

Cluster 1 (size=9, mean_prob=0.9882):
_fecha, .flutter, _producto, _firestore, _empresa, _produk, _vue, _barang, /provider

Cluster 2 (size=5, mean_prob=1.0000):
_Tis, .JOption, )가, )은, \Mapping

Cluster 3 (size=24, mean_prob=0.8991):
	dp, 	mt, 	fd, 	sf, 	td, 	pm, 	pp, 	rt, 	fp, 	md, 	mp, 	cc, 	ds, 	pc, 	lp, 	ct, 	tc, 	ws, 	cv, 	vm

Cluster 4 (size=33, mean_prob=0.9373):
“There, “All, “If, “He, “And, “My, “That, “As, “When, “To, “She, “At, “How, “One, “Yes, (numero, “For, “Our, “You, “They

Cluster 5 (size=6, mean_prob=0.9989):
 beiden,  eigenen,  richtigen,  entsprechenden,  einfachen,  höheren

Cluster 6 (size=9, mean_prob=0.9997):
"D, "T, "M, "W, "L, "E, "G, "P, "S

Cluster 7 (size=5, mean_prob=1.0000):
ュ, ？
, ョ, ーク, ープ

Cluster 8 (size=9, mean_prob=0.9773):
'év, 'intérêt, 'entrée, 'accès, 'activité, 'énergie, 'évolution, 'entreprise, 'améli

Cluster 9 (size=6, mean

In [16]:
print_from_csv("comp/gpt-oss/output_proj/random/seed_0/clusters_l2=True/run_00001_clusters.csv", tokenizer)

Cluster 0 (size=5, mean_prob=1.0000):
/repos, /problems, /avatar, /gallery, .king

Cluster 1 (size=6, mean_prob=0.9971):
-dd, -fw, -mm, -aa, /oct, -ie

Cluster 2 (size=20, mean_prob=0.6812):
	F, 	M, 	D, 	G, 	R, 	L, 	H, 	P, 	T, 	B, 	V, 	N, 	E, 	A, 	I, 	C, 	S, 	U, 	K, 	O

Cluster 3 (size=11, mean_prob=0.9876):
’E, ’O, ’H, .bluetooth, ’U, ’I, ®,, ’A, .gwt, .cloudflare, .userid

Cluster 4 (size=6, mean_prob=0.9974):
_producto, _fecha, _empresa, _firestore, _produk, .kode

Cluster 5 (size=27, mean_prob=0.7354):
П, К, Н, Т, З, У, М, Д, А, Г, О, Ф, Я, И, Х, Р, Б, Ш, Л, Ч

Cluster 6 (size=12, mean_prob=0.9355):
, , , , , , , , , , )+(, ￼

Cluster 7 (size=10, mean_prob=0.9907):
"M, "D, "T, "G, "W, "L, "E, "H, "P, "S

Cluster 8 (size=8, mean_prob=0.9989):
*)), //------------------------------------------------, =====
, +</, $tmp, &&!, =>', //--------------------------------

Cluster 9 (size=19, mean_prob=0.9306):
	mp, 	md, 	sf, 	td, 	pp, 	mt, 	fp, 	pm, 	rt, 	dp, 	fd, 	cc, 	ws, 	tc, 	ct

In [12]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/random/seed_0/clusters_l2=True/run_00001_clusters.csv", tokenizer)

Cluster 0 (size=10, mean_prob=0.9607):
со, до, 如, mathrm, mathbf, 在, mbox, з, ко, ка

Cluster 1 (size=5, mean_prob=1.0000):
foreach, tf, grin, slid, wiped

Cluster 2 (size=6, mean_prob=0.9993):
constants, novels, provin, graphs, chunks, headers

Cluster 3 (size=5, mean_prob=1.0000):
Nel, Ges, Dans, Bere, Gott

Cluster 4 (size=5, mean_prob=1.0000):
espec, ü, ú, anche, onder

Cluster 5 (size=16, mean_prob=0.9460):
José, Sé, Herz, Zw, Kont, Garc, Mais, Geb, Graf, Schwe, Zeit, Juni, Mé, Pé, Hamb, Dé

Cluster 6 (size=5, mean_prob=1.0000):
marry, pd, disg, compile, homosexual

Cluster 7 (size=6, mean_prob=0.9959):
agram, atta, kle, oka, vil, onclick

Cluster 8 (size=5, mean_prob=1.0000):
jest, gorge, foo, dick, tir

Cluster 9 (size=15, mean_prob=0.9266):
elem, agric, ignor, olymp, aws, iterator, assh, osc, encode, amplit, awk, onChange, embar, sd, initialize

Cluster 10 (size=5, mean_prob=1.0000):
Su, Cell, Press, Je, Board

Cluster 11 (size=7, mean_prob=0.9988):
Album, Auth, Iter, Agreement

In [13]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/random/seed_42/clusters_l2=True/run_00001_clusters.csv", tokenizer)

Cluster 0 (size=5, mean_prob=1.0000):
lapt, satell, navigate, calories, fighters

Cluster 1 (size=10, mean_prob=0.9612):
со, до, 如, mathrm, mathbf, 在, з, mbox, ко, ка

Cluster 2 (size=6, mean_prob=0.9990):
headers, constants, novels, provin, chunks, graphs

Cluster 3 (size=5, mean_prob=1.0000):
suc, mars, neutr, sep, litt

Cluster 4 (size=5, mean_prob=1.0000):
vertex, router, hurried, hurry, decimal

Cluster 5 (size=5, mean_prob=1.0000):
espec, ü, ú, anche, onder

Cluster 6 (size=14, mean_prob=0.9826):
Zeit, José, Mais, Garc, Herz, Zw, Sé, Kont, Graf, Geb, Juni, Pé, Mé, Schwe

Cluster 7 (size=5, mean_prob=1.0000):
marry, pd, disg, compile, homosexual

Cluster 8 (size=5, mean_prob=1.0000):
ird, aron, hs, sex, hh

Cluster 9 (size=15, mean_prob=0.9092):
elem, agric, ignor, aws, iterator, olymp, assh, osc, amplit, awk, sd, embar, onChange, encode, initialize

Cluster 10 (size=9, mean_prob=0.9974):
Catalogue, Municip, Parse, Logger, Reserved, Verify, Widget, Describe, Parameter

Cluster 11 

In [3]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00003_clusters.csv", tokenizer)

Cluster 0 (size=13, mean_prob=0.9427):
Scale, Scal, Scal, scaling, scales, scal, scale, scalar, scal, Scalar, scalar, scale, 缩

Cluster 1 (size=12, mean_prob=0.9759):
июня, ноября, октября, июля, февраля, декабря, сентября, апреля, января, марта, августа, мая

Cluster 2 (size=7, mean_prob=0.9904):
tab, Tab, tab, tabular, Tab, tabs, TabIndex

Cluster 3 (size=11, mean_prob=0.9459):
tag, Tag, tag, Tag, tags, TAG, Tags, tags, 태, 标, タ

Cluster 4 (size=20, mean_prob=0.9590):
parameter, parameters, parameter, parameters, 参, Parameter, Param, param, Param, Parameters, params, param, Parameter, Params, Parameters, params, PARAM, PARAMETER, 參, 참

Cluster 5 (size=7, mean_prob=0.9683):
orient, orient, orientation, Orient, Ori, oriented, rient

Cluster 6 (size=5, mean_prob=1.0000):
anch, anche, Anchor, anchor, anch

Cluster 7 (size=15, mean_prob=0.9651):
ления, жения, чения, нения, шения, нение, ление, ждения, вания, жение, чение, ження, шение, лення, чення

Cluster 8 (size=8, mean_prob=0.9666):
mi

In [4]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00005_clusters.csv", tokenizer)

Cluster 0 (size=13, mean_prob=0.9548):
Service, service, service, Services, Service, services, services, Services, SERVICE, ervice, служ, ervices, слу

Cluster 1 (size=7, mean_prob=0.9917):
tab, Tab, tab, tabular, Tab, tabs, TabIndex

Cluster 2 (size=8, mean_prob=0.9773):
band, band, Band, bands, Band, bands, banda, 带

Cluster 3 (size=17, mean_prob=0.9728):
ления, нения, жения, чения, шения, жение, чение, нение, ждения, ление, шение, вания, ження, лення, чення, zenia, вання

Cluster 4 (size=8, mean_prob=0.9805):
sq, square, sq, Square, square, sqrt, squ, quare

Cluster 5 (size=9, mean_prob=0.9416):
channel, Channel, channel, Channel, channels, channels, CHANNEL, annels, chan

Cluster 6 (size=8, mean_prob=0.9689):
root, root, Root, roots, ROOT, Root, 根, Based

Cluster 7 (size=7, mean_prob=0.9950):
ster, sters, стер, STER, ster, Ster, сте

Cluster 8 (size=10, mean_prob=0.9655):
Chapter, chapter, Chap, chap, chapters, chap, CHAPTER, apter, 章, APTER

Cluster 9 (size=6, mean_prob=1.0000):
a

In [5]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00006_clusters.csv", tokenizer)

Cluster 0 (size=13, mean_prob=0.9651):
Service, service, service, Services, Service, services, Services, SERVICE, services, ervice, служ, ervices, слу

Cluster 1 (size=10, mean_prob=0.9433):
Engine, Engine, engineers, engine, engineer, engineering, Engineering, engine, engines, 엔

Cluster 2 (size=8, mean_prob=0.9633):
Dele, deleg, Delegate, deleg, delegate, delegate, 委, deck

Cluster 3 (size=16, mean_prob=0.9774):
ления, жения, чения, шения, нения, жение, чение, нение, вания, шение, ление, ждения, ження, лення, чення, zenia

Cluster 4 (size=7, mean_prob=0.9930):
tab, Tab, tab, tabular, Tab, tabs, TabIndex

Cluster 5 (size=8, mean_prob=0.9793):
band, band, Band, bands, Band, bands, banda, 带

Cluster 6 (size=9, mean_prob=0.9435):
channel, Channel, channel, Channel, channels, channels, CHANNEL, annels, chan

Cluster 7 (size=16, mean_prob=0.9085):
check, Check, Check, checked, checking, check, checks, CHECK, CHECK, checked, 检, Checked, Checker, checkbox, chk, 체

Cluster 8 (size=5, mean_pro

In [7]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00007_clusters.csv", tokenizer)

Cluster 0 (size=10, mean_prob=0.9445):
Engine, Engine, engineers, engine, engineer, engineering, Engineering, engine, engines, 엔

Cluster 1 (size=13, mean_prob=0.9678):
Service, service, service, Services, Service, services, Services, services, ervice, SERVICE, служ, ervices, слу

Cluster 2 (size=6, mean_prob=1.0000):
ὸ, ῶ, ὰ, ῆ, ὴ, ὶ

Cluster 3 (size=16, mean_prob=0.9787):
ления, жения, чения, шения, нения, жение, чение, нение, ление, шение, вания, ждения, ження, лення, чення, zenia

Cluster 4 (size=8, mean_prob=0.9649):
Dele, deleg, Delegate, deleg, delegate, delegate, 委, deck

Cluster 5 (size=9, mean_prob=0.9442):
channel, Channel, channel, Channel, channels, channels, CHANNEL, annels, chan

Cluster 6 (size=16, mean_prob=0.9105):
check, Check, Check, checked, checking, check, checks, CHECK, CHECK, checked, 检, Checked, Checker, checkbox, chk, 체

Cluster 7 (size=5, mean_prob=1.0000):
anch, anches, anchor, Anchor, anchor

Cluster 8 (size=7, mean_prob=0.9961):
agg, aggreg, ggreg, aggreg

In [6]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00008_clusters.csv", tokenizer)

Cluster 0 (size=8, mean_prob=0.9572):
device, device, Device, devices, Device, devices, DEVICE, 裝

Cluster 1 (size=10, mean_prob=0.9447):
Engine, Engine, engineers, engine, engineer, engineering, Engineering, engine, engines, 엔

Cluster 2 (size=13, mean_prob=0.9673):
Service, service, service, Services, Service, services, Services, services, ervice, SERVICE, служ, ervices, слу

Cluster 3 (size=7, mean_prob=0.9995):
ὸ, ῆ, ῶ, ὰ, ὴ, ὶ, ή

Cluster 4 (size=16, mean_prob=0.9779):
ления, жения, чения, шения, нения, жение, чение, нение, ление, шение, вания, ждения, ження, лення, чення, zenia

Cluster 5 (size=8, mean_prob=0.9651):
Dele, deleg, Delegate, deleg, delegate, delegate, 委, deck

Cluster 6 (size=16, mean_prob=0.9103):
check, Check, Check, checked, checking, check, checks, CHECK, CHECK, checked, 检, Checked, Checker, checkbox, chk, 체

Cluster 7 (size=9, mean_prob=0.9444):
channel, Channel, channel, Channel, channels, channels, CHANNEL, annels, chan

Cluster 8 (size=8, mean_prob=0.9802):


In [6]:
print_from_csv("comp/mistralai/Mistral-7B-v0.1/output_proj/clusters_l2=True/run_00009_clusters.csv", tokenizer)

Cluster 0 (size=8, mean_prob=0.9573):
device, device, Device, devices, Device, devices, DEVICE, 裝

Cluster 1 (size=10, mean_prob=0.9447):
Engine, Engine, engineers, engine, engineer, engineering, Engineering, engine, engines, 엔

Cluster 2 (size=13, mean_prob=0.9673):
Service, service, service, Services, Service, services, Services, services, ervice, SERVICE, служ, ervices, слу

Cluster 3 (size=7, mean_prob=0.9995):
ὸ, ῆ, ῶ, ὰ, ὴ, ὶ, ή

Cluster 4 (size=16, mean_prob=0.9780):
ления, жения, чения, шения, нения, жение, чение, нение, ление, шение, вания, ждения, ження, лення, чення, zenia

Cluster 5 (size=8, mean_prob=0.9651):
Dele, deleg, Delegate, deleg, delegate, delegate, 委, deck

Cluster 6 (size=16, mean_prob=0.9103):
check, Check, Check, checked, checking, check, checks, CHECK, CHECK, checked, 检, Checked, Checker, checkbox, chk, 체

Cluster 7 (size=9, mean_prob=0.9444):
channel, Channel, channel, Channel, channels, channels, CHANNEL, annels, chan

Cluster 8 (size=8, mean_prob=0.9802):
